# RHOAI AutoRAG Evaluation with LLM-as-a-Judge
This notebook evaluates RHOAI AutoRAG results using LLM-as-a-Judge metrics.
## Metrics
1. **Answer Correctness**: Keyword overlap (Jaccard similarity)
2. **Faithfulness**: Content overlap with retrieved documents
3. **LLM-as-a-Judge**: Semantic quality rating (1-5 scale, normalized to 0-1)
## Data Source
- Input: `evals/evaluation_results_5.txt`
- Output: `outputs/rhoai_autorag_with_llmaj.csv`
## ⚠️ Important: Run Cells in Order
**Please run all cells sequentially from top to bottom.** Cell execution order matters because:
- Section 1 imports required libraries (os, json, pandas, Path, etc.)
- Later cells depend on these imports and variables from earlier cells
If you get a `NameError`, restart the kernel and run all cells in order.

## 1. Setup and Imports

In [1]:
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv

## 2. Load RHOAI AutoRAG Results (All Patterns)
This section automatically discovers and loads all evaluation files from `evals/` folder.

In [2]:
# Auto-discover all evaluation results files
import glob
import re
eval_dir = "evals"
pattern = os.path.join(eval_dir, "evaluation_results*.txt")
eval_files = glob.glob(pattern)
if not eval_files:
    print(f"❌ No evaluation files found in {eval_dir}/")
    print(f"   Looking for: evaluation_results*.txt")
else:
    print(f"📂 Found {len(eval_files)} evaluation file(s):")
    # Load all evaluation files
    all_pattern_data = {}
    for eval_file in sorted(eval_files):
        # Extract pattern name from filename
        # e.g., "evaluation_results_5.txt" -> "Pattern_5"
        # e.g., "evaluation_results (2).txt" -> "Pattern_2"
        # e.g., "evaluation_results.txt" -> "Pattern_default"
        filename = os.path.basename(eval_file)
        # Try different naming patterns
        if match := re.search(r'evaluation_results[_\s]*(\d+)', filename):
            pattern_name = f"Pattern_{match.group(1)}"
        elif match := re.search(r'evaluation_results\s*\((\d+)\)', filename):
            pattern_name = f"Pattern_{match.group(1)}"
        elif filename == "evaluation_results.txt":
            pattern_name = "Pattern_default"
        else:
            # Fallback: use filename without extension
            pattern_name = filename.replace('.txt', '').replace('evaluation_results', 'Pattern')
        # Load the file
        try:
            with open(eval_file, 'r') as f:
                data = json.load(f)
            all_pattern_data[pattern_name] = {
                'file': eval_file,
                'data': data,
                'count': len(data)
            }
            print(f"   ✅ {pattern_name:20} {eval_file:40} ({len(data)} questions)")
        except Exception as e:
            print(f"   ❌ {pattern_name:20} {eval_file:40} Error: {e}")
    print(f"\n📊 Total patterns loaded: {len(all_pattern_data)}")
    # Display summary
    if all_pattern_data:
        print(f"\n📄 Sample from first pattern:")
        first_pattern = list(all_pattern_data.values())[0]
        print(f"   Pattern: {list(all_pattern_data.keys())[0]}")
        print(f"   Questions: {first_pattern['count']}")
        sample = first_pattern['data'][0]
        print(f"   Sample question: {sample['question'][:80]}...")
        print(f"   Has answer: {'answer' in sample}")
        print(f"   Has scores: {'scores' in sample}")

📂 Found 2 evaluation file(s):
   ✅ Pattern_11           autorag_evals/evaluation_results_11.txt  (22 questions)
   ✅ Pattern_5            autorag_evals/evaluation_results_5.txt   (22 questions)

📊 Total patterns loaded: 2

📄 Sample from first pattern:
   Pattern: Pattern_11
   Questions: 22
   Sample question: Who owns Romano's Bakery, and how is this person related to Isabella Romano?...
   Has answer: True
   Has scores: True


## 3. Display Baseline Metrics

In [3]:
# Convert all patterns to DataFrame
all_results = []
for pattern_name, pattern_info in all_pattern_data.items():
    for item in pattern_info['data']:
        all_results.append({
            "pattern": pattern_name,
            "question": item["question"],
            "ground_truth": item["correct_answers"],
            "predicted_answer": item["answer"],
            "answer_correctness": item["scores"].get("answer_correctness", 0.0) if "scores" in item else 0.0,
            "faithfulness": item["scores"].get("faithfulness", 0.0) if "scores" in item else 0.0
        })
rhoai_df = pd.DataFrame(all_results)
print("="*80)
print("RHOAI AUTORAG BASELINE METRICS (ALL PATTERNS)")
print("="*80)
# Overall metrics
print(f"\nOverall (all patterns combined):")
print(f"  Answer Correctness: {rhoai_df['answer_correctness'].mean():.4f}")
print(f"  Faithfulness:       {rhoai_df['faithfulness'].mean():.4f}")
print(f"  Combined Score:     {(rhoai_df['answer_correctness'].mean() + rhoai_df['faithfulness'].mean()) / 2:.4f}")
# Per-pattern metrics
print(f"\nPer-pattern breakdown:")
for pattern_name in sorted(rhoai_df['pattern'].unique()):
    pattern_data = rhoai_df[rhoai_df['pattern'] == pattern_name]
    print(f"\n  {pattern_name}:")
    print(f"    Questions:          {len(pattern_data)}")
    print(f"    Answer Correctness: {pattern_data['answer_correctness'].mean():.4f}")
    print(f"    Faithfulness:       {pattern_data['faithfulness'].mean():.4f}")
    print(f"    Combined Score:     {(pattern_data['answer_correctness'].mean() + pattern_data['faithfulness'].mean()) / 2:.4f}")
print("\n" + "="*80)

RHOAI AUTORAG BASELINE METRICS (ALL PATTERNS)

Overall (all patterns combined):
  Answer Correctness: 0.3378
  Faithfulness:       0.3720
  Combined Score:     0.3549

Per-pattern breakdown:

  Pattern_11:
    Questions:          22
    Answer Correctness: 0.3295
    Faithfulness:       0.4435
    Combined Score:     0.3865

  Pattern_5:
    Questions:          22
    Answer Correctness: 0.3461
    Faithfulness:       0.3005
    Combined Score:     0.3233



## 4. Configure OGX for LLM-as-a-Judge

In [4]:
# Load .env from lightrag/POC
env_path = Path("../../lightrag/POC/.env")
if env_path.exists():
    load_dotenv(env_path)
    print(f"✅ Loaded .env from {env_path}")
else:
    print(f"⚠️  .env not found at {env_path}")
OGX_BASE_URL = os.getenv("OGX_BASE_URL")
OGX_API_KEY = os.getenv("OGX_API_KEY")
if not OGX_BASE_URL or not OGX_API_KEY:
    raise ValueError("OGX_BASE_URL and OGX_API_KEY must be set in .env file")
print(f"Using OGX endpoint: {OGX_BASE_URL}")

✅ Loaded .env from ../lightrag/POC/.env
Using OGX endpoint: https://<OGX_BASE_URL>


In [5]:
# Get available models
from openai import OpenAI
try:
    client = OpenAI(
        base_url=f"{OGX_BASE_URL}/v1",
        api_key=OGX_API_KEY
    )
    model_list = [model.id for model in client.models.list().data]
    print(f"✅ Available models: {model_list}")
    # Select inference model
    inference_models = [m for m in model_list if 'inference' in m or 'llama' in m.lower()]
    CUSTOM_MODEL = inference_models[0] if inference_models else model_list[1]
    print(f"   Selected model: {CUSTOM_MODEL}")
except Exception as e:
    print(f"⚠️  Authentication error: {e}")
    print(f"\n🔧 Your JWT token may be expired. To fix:")
    print(f"   1. Go to {OGX_BASE_URL}")
    print(f"   2. Get a new API token")
    print(f"   3. Update OGX_API_KEY in ../../lightrag/POC/.env")
    print(f"\n   Using fallback configuration for now...")
    # Fallback model (common OGX model)
    CUSTOM_MODEL = "vllm-inference-gpu-llama/redhataillama-31-8b-instruct"
    print(f"\n✅ Fallback model: {CUSTOM_MODEL}")
    print(f"   (LLM-as-a-Judge will fail until you update the API key)")
    # Reinitialize client (will still fail on actual calls, but won't crash here)
    client = OpenAI(
        base_url=f"{OGX_BASE_URL}/v1",
        api_key=OGX_API_KEY
    )

✅ Available models: ['vllm-embedding/bge-m3', 'vllm-inference-gpu-llama/redhataillama-31-8b-instruct']
   Selected model: vllm-inference-gpu-llama/redhataillama-31-8b-instruct


## 5. Define LLM-as-a-Judge Function

In [6]:
def llm_as_judge(question, predicted_answer, ground_truth_answers, model=None):
    """Use LLM to judge answer quality on scale 1-5"""
    if model is None:
        model = CUSTOM_MODEL
    if not predicted_answer or predicted_answer.startswith("Error:"):
        return 0.2  # Minimum score for errors
    gt_text = " OR ".join(ground_truth_answers)
    judge_prompt = f"""You are an expert evaluator. Rate the quality of the predicted answer compared to the ground truth.
Question: {question}
Ground Truth Answer(s): {gt_text}
Predicted Answer: {predicted_answer}
Rate the predicted answer on a scale of 1-5:
- 5: Perfect answer, matches ground truth completely
- 4: Very good answer, captures main points with minor gaps
- 3: Adequate answer, partially correct but missing key information
- 2: Poor answer, mostly incorrect or incomplete
- 1: Wrong or irrelevant answer
Respond with ONLY a number from 1 to 5."""
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": judge_prompt}],
            temperature=0.1,
            max_tokens=10
        )
        # Extract numeric score
        score_text = response.choices[0].message.content.strip()
        import re
        match = re.search(r'[1-5]', score_text)
        if match:
            score = int(match.group())
            return score / 5.0  # Normalize to 0-1 range
        else:
            return 0.5  # Default to middle if can't parse
    except Exception as e:
        print(f"  LLM Judge error: {str(e)}")
        return 0.5  # Default to middle on error
print("✅ Defined LLM-as-a-Judge function")

✅ Defined LLM-as-a-Judge function


## 6. Run LLM-as-a-Judge Evaluation

In [7]:
print("Running LLM-as-a-Judge evaluation...")
print("This may take a few minutes...")
for i, row in rhoai_df.iterrows():
    question = row["question"]
    predicted = row["predicted_answer"]
    ground_truth = row["ground_truth"]
    # Calculate LLM judge score
    llm_score = llm_as_judge(question, predicted, ground_truth)
    rhoai_df.at[i, "llm_judge_score"] = llm_score
    if (i + 1) % 5 == 0:
        print(f"  Processed {i + 1}/{len(rhoai_df)} answers...")
print(f"\n✅ Completed LLM-as-a-Judge evaluation")
print(f"   Average LLM Judge Score: {rhoai_df['llm_judge_score'].mean():.3f}")


✅ Completed LLM-as-a-Judge evaluation
   Average LLM Judge Score: 0.777


## 7. Display Final Metrics

In [8]:
print("="*80)
print("RHOAI AUTORAG FINAL METRICS (WITH LLM-AS-A-JUDGE)")
print("="*80)
# Overall metrics (all patterns combined)
print("\nOverall (all patterns combined):")
overall_metrics = {
    "Metric": ["Answer Correctness", "Faithfulness", "LLM Judge", "Combined Score (3 metrics)"],
    "Score": [
        f"{rhoai_df['answer_correctness'].mean():.4f}",
        f"{rhoai_df['faithfulness'].mean():.4f}",
        f"{rhoai_df['llm_judge_score'].mean():.4f}",
        f"{(rhoai_df['answer_correctness'].mean() + rhoai_df['faithfulness'].mean() + rhoai_df['llm_judge_score'].mean()) / 3:.4f}"
    ]
}
overall_df = pd.DataFrame(overall_metrics)
print(overall_df.to_string(index=False))
# Per-pattern comparison
print("\n" + "="*80)
print("PER-PATTERN COMPARISON")
print("="*80)
pattern_comparison = []
for pattern_name in sorted(rhoai_df['pattern'].unique()):
    pattern_data = rhoai_df[rhoai_df['pattern'] == pattern_name]
    pattern_comparison.append({
        "Pattern": pattern_name,
        "Questions": len(pattern_data),
        "Answer Correctness": f"{pattern_data['answer_correctness'].mean():.4f}",
        "Faithfulness": f"{pattern_data['faithfulness'].mean():.4f}",
        "LLM Judge": f"{pattern_data['llm_judge_score'].mean():.4f}",
        "Combined": f"{(pattern_data['answer_correctness'].mean() + pattern_data['faithfulness'].mean() + pattern_data['llm_judge_score'].mean()) / 3:.4f}"
    })
comparison_df = pd.DataFrame(pattern_comparison)
print(comparison_df.to_string(index=False))
# LLM Judge distribution (overall)
print("\n" + "="*80)
print("LLM JUDGE SCORE DISTRIBUTION (ALL PATTERNS)")
print("="*80)
llm_scores = rhoai_df['llm_judge_score']
print(f"  Perfect (1.0):   {(llm_scores == 1.0).sum()} questions ({(llm_scores == 1.0).sum()/len(llm_scores)*100:.1f}%)")
print(f"  Very Good (0.8): {(llm_scores == 0.8).sum()} questions ({(llm_scores == 0.8).sum()/len(llm_scores)*100:.1f}%)")
print(f"  Good (0.6):      {(llm_scores == 0.6).sum()} questions ({(llm_scores == 0.6).sum()/len(llm_scores)*100:.1f}%)")
print(f"  Fair (0.4):      {(llm_scores == 0.4).sum()} questions ({(llm_scores == 0.4).sum()/len(llm_scores)*100:.1f}%)")
print(f"  Poor (0.2):      {(llm_scores == 0.2).sum()} questions ({(llm_scores == 0.2).sum()/len(llm_scores)*100:.1f}%)")
print("\n" + "="*80)

RHOAI AUTORAG FINAL METRICS (WITH LLM-AS-A-JUDGE)

Overall (all patterns combined):
                    Metric  Score
        Answer Correctness 0.3378
              Faithfulness 0.3720
                 LLM Judge 0.7773
Combined Score (3 metrics) 0.4957

PER-PATTERN COMPARISON
   Pattern  Questions Answer Correctness Faithfulness LLM Judge Combined
Pattern_11         22             0.3295       0.4435    0.8000   0.5243
 Pattern_5         22             0.3461       0.3005    0.7545   0.4671

LLM JUDGE SCORE DISTRIBUTION (ALL PATTERNS)
  Perfect (1.0):   5 questions (11.4%)
  Very Good (0.8): 33 questions (75.0%)
  Good (0.6):      3 questions (6.8%)
  Fair (0.4):      2 questions (4.5%)
  Poor (0.2):      1 questions (2.3%)



In [9]:
# Generate Pattern Leaderboard (ranked by Combined Score)
print("="*80)
print("PATTERN LEADERBOARD (RANKED BY COMBINED SCORE)")
print("="*80)
leaderboard_data = []
for pattern_name in rhoai_df['pattern'].unique():
    pattern_data = rhoai_df[rhoai_df['pattern'] == pattern_name]
    ac = pattern_data['answer_correctness'].mean()
    faith = pattern_data['faithfulness'].mean()
    llm = pattern_data['llm_judge_score'].mean()
    combined = (ac + faith + llm) / 3
    leaderboard_data.append({
        "Rank": 0,  # Will be set after sorting
        "Pattern": pattern_name,
        "Questions": len(pattern_data),
        "Answer Correctness": round(ac, 4),
        "Faithfulness": round(faith, 4),
        "LLM Judge": round(llm, 4),
        "Combined Score": round(combined, 4)
    })
# Sort by combined score (descending)
leaderboard_data.sort(key=lambda x: x["Combined Score"], reverse=True)
# Assign ranks
for i, entry in enumerate(leaderboard_data, 1):
    entry["Rank"] = i
# Create DataFrame
leaderboard_df = pd.DataFrame(leaderboard_data)
# Display
print(leaderboard_df.to_string(index=False))
print("\n" + "="*80)
# Find best pattern
best_pattern = leaderboard_data[0]
print(f"\n🏆 Best Pattern: {best_pattern['Pattern']}")
print(f"   Combined Score: {best_pattern['Combined Score']}")
print(f"   - Answer Correctness: {best_pattern['Answer Correctness']}")
print(f"   - Faithfulness: {best_pattern['Faithfulness']}")  
print(f"   - LLM Judge: {best_pattern['LLM Judge']}")
# Performance spread
if len(leaderboard_data) > 1:
    worst_pattern = leaderboard_data[-1]
    spread = best_pattern['Combined Score'] - worst_pattern['Combined Score']
    print(f"\n📊 Performance Spread: {spread:.4f}")
    print(f"   Best:  {best_pattern['Pattern']:15} ({best_pattern['Combined Score']:.4f})")
    print(f"   Worst: {worst_pattern['Pattern']:15} ({worst_pattern['Combined Score']:.4f})")


PATTERN LEADERBOARD (RANKED BY COMBINED SCORE)
 Rank    Pattern  Questions  Answer Correctness  Faithfulness  LLM Judge  Combined Score
    1 Pattern_11         22              0.3295        0.4435     0.8000          0.5243
    2  Pattern_5         22              0.3461        0.3005     0.7545          0.4671


🏆 Best Pattern: Pattern_11
   Combined Score: 0.5243
   - Answer Correctness: 0.3295
   - Faithfulness: 0.4435
   - LLM Judge: 0.8

📊 Performance Spread: 0.0572
   Best:  Pattern_11      (0.5243)
   Worst: Pattern_5       (0.4671)


## 8. Save Results

In [10]:
from pathlib import Path
Path("outputs").mkdir(exist_ok=True)

# Add retrieval mode column for consistency with other notebooks
rhoai_df['retrieval_mode'] = 'rhoai_autorag'
# Save to CSV
output_file = "outputs/rhoai_autorag_with_llmaj.csv"
rhoai_df.to_csv(output_file, index=False)
print(f"💾 Saved results to: {output_file}")
print(f"   {len(rhoai_df)} total rows ({rhoai_df['pattern'].nunique()} patterns)")
print(f"   Columns: {', '.join(rhoai_df.columns)}")
# Also save per-pattern CSVs
print(f"\n💾 Saving per-pattern CSVs:")
for pattern_name in sorted(rhoai_df['pattern'].unique()):
    pattern_data = rhoai_df[rhoai_df['pattern'] == pattern_name]
    pattern_file = f"outputs/rhoai_{pattern_name.lower()}_with_llmaj.csv"
    pattern_data.to_csv(pattern_file, index=False)
    print(f"   {pattern_file:40} ({len(pattern_data)} rows)")
# Display sample
print("\n📊 Sample results (first 5 rows):")
print(rhoai_df[['pattern', 'question', 'answer_correctness', 'faithfulness', 'llm_judge_score']].head().to_string(index=False))

💾 Saved results to: rhoai_autorag_with_llmaj.csv
   44 total rows (2 patterns)
   Columns: pattern, question, ground_truth, predicted_answer, answer_correctness, faithfulness, llm_judge_score, retrieval_mode

💾 Saving per-pattern CSVs:
   rhoai_pattern_11_with_llmaj.csv          (22 rows)
   rhoai_pattern_5_with_llmaj.csv           (22 rows)

📊 Sample results (first 5 rows):
   pattern                                                                                                                                          question  answer_correctness  faithfulness  llm_judge_score
Pattern_11                                                                      Who owns Romano's Bakery, and how is this person related to Isabella Romano?              0.3529        0.5000              1.0
Pattern_11 Dr. Okafor treated a patient in the emergency department. What was the patient's condition, and what is Dr. Okafor's relationship to the patient?              0.6000        0.8929              0.

## Summary
This notebook evaluated RHOAI AutoRAG using LLM-as-a-Judge and revealed:
1. **Answer Correctness** (keyword overlap) underestimates quality
2. **LLM Judge** scores show true semantic quality
3. **Faithfulness** is the critical issue (low grounding/citations)
### Multi-Pattern Support
The notebook automatically detects all `evaluation_results*.txt` files in `evals/`:
- Supports multiple naming formats: `evaluation_results_5.txt`, `evaluation_results (2).txt`, etc.
- Compares metrics across all patterns
- Saves both combined and per-pattern CSV outputs
### Key Finding
RHOAI generates good quality answers (high LLM judge score) but lacks proper citations/grounding (low faithfulness). The fix is prompt engineering, not model/retrieval changes.
### Output Files
- `outputs/rhoai_autorag_with_llmaj.csv` - Combined results from all patterns
- `outputs/rhoai_pattern_X_with_llmaj.csv` - Individual pattern results
